In [ ]:
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset

import torchmetrics

import torchvision
import torchvision.transforms as transforms
from torchvision import datasets

import optuna

In [26]:
# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [27]:
class CNN(nn.Module):
    def __init__(self, n_layers, n_filters, kernel_sizes, dropout_rate, fc_size):
        super(CNN, self).__init__()

        # initilize an empty list to hold convolutional blocks
        blocks = []

        # the intitilization of input channels for RGB images
        in_channels = 3

        # loop in construct each block
        for i in range(n_layers):
            # get the parameters of the current layer
            out_channels = n_filters[i]
            kernel_size = kernel_sizes[i]
            # calculate padding
            padding = (kernel_size - 1) // 2

            block = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2, stride=2)
            )
            blocks.append(block)

            # Update the number of input channels for the next block
            in_channels = out_channels

        # combine the blocks into one extractor module
        self.features = nn.Sequential(*blocks)

        # Store hyperparameters needed for building the classifier
        self.dropout_rate = dropout_rate
        self.fc_size = fc_size

        # The classifier will be initialized dynamically in the forward pass
        self.classifier = None

    def _create_classifier(self, flattened_size, device):
        # Define the classifier's architecture
        self.classifier = nn.Sequential(
            nn.Dropout(self.dropout_rate),
            nn.Linear(flattened_size, self.fc_size),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.fc_size, 10)             # 10 output classes (CIFAR-10)
        ).to(device)

    def forward(self, x):
        # Get the device of the input tensor to ensure consistency
        device = x.device

        # pass the input through the feature extraction layer
        x = self.features(x)

        # Flatten the feature map to prepare it for the fully connected layers
        flattened = torch.flatten(x, 1)
        flattened_size = flattened.size(1)

        # If the classifier has not been created yet, initialize it
        if self.classifier is None:
            self._create_classifier(flattened_size, device)

        # Pass the flattened features through the classifier to get the final output
        return self.classifier(flattened)

In [ ]:
def get_data_loaders_with_validation(batch_size, val_fraction=0.1):
    """Creates and returns data loaders for training, validation, and testing.

    Args:
        batch_size: The number of samples per batch in each data loader.
        val_fraction: The fraction of the training data to use for validation.

    Returns:
        A tuple containing the training, validation, and test data loaders.
    """
    # Define the transformations for the training data, including augmentation.
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    # Define the transformations for the validation and test data (no augmentation).
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    # Load the full CIFAR-10 training dataset with the TRAIN transform.
    full_trainset = datasets.CIFAR10(root='./cifar10', train=True, download=True, transform=transform_train)
    
    # Load the full CIFAR-10 training dataset again with the TEST transform (for validation).
    # You need a separate object so the validation data doesn't get augmented.
    full_valset = datasets.CIFAR10(root='./cifar10', train=True, download=True, transform=transform_test)

    # Calculate the number of samples for the training and validation sets.
    total_train = len(full_trainset)
    val_size = int(val_fraction * total_train)
    train_size = total_train - val_size

    # Perform the split to generate random indices. 
    # You use full_trainset to generate the split, but you will apply indices to the correct backends below.
    train_subset_temp, val_subset_temp = random_split(full_trainset, [train_size, val_size])

    # Create the final train_set and val_set using the specific parent datasets and the generated indices.
    # This ensures train_set uses 'full_trainset' (augmented) and val_set uses 'full_valset' (not augmented).
    train_set = Subset(full_trainset, train_subset_temp.indices)
    val_set = Subset(full_valset, val_subset_temp.indices)

    # Load the CIFAR-10 test dataset.
    test_set = datasets.CIFAR10(root='./cifar10', train=False, download=True, transform=transform_test)

    # Create DataLoader instances for each dataset split.
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2)

    # Return the created data loaders.
    return train_loader, val_loader, test_loader

In [ ]:
"""""
def train_model(model, optimizer, train_dataloader, n_epochs, loss_fcn, device):
    model.train()  # Set model to training mode

    for epoch in range(n_epochs):
        running_loss = 0.0
        total = 0
        correct = 0

        for inputs, labels in train_dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()
            # Forward pass and
            outputs = model(inputs)
            loss = loss_fcn(outputs, labels)
            # Backward pass and optimization
            loss.backward()
            optimizer.step()

            # Accumulate statistics
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_loss = running_loss / total
        epoch_acc = correct / total
        print(f"Epoch [{epoch + 1}/{n_epochs}] - Loss: {epoch_loss:.4f} - Accuracy: {epoch_acc:.4f}")

    return model
"""""

In [ ]:
@torch.no_grad()
def evaluate_model(model, dataloader, device):
    model.eval()
    total = 0
    correct = 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return correct / total

In [ ]:
def train_and_evaluate(model, optimizer, scheduler, device, n_epochs, train_loader, val_loader):
    """Trains a model for a number of epochs, evaluating on the validation set
    after each epoch, and steps a learning rate scheduler.

    Args:
        model: The neural network to train.
        optimizer: The optimizer used to update model parameters.
        scheduler: A learning rate scheduler (e.g. StepLR, CosineAnnealingLR, etc.).
        device: The device to run training on ('cuda' or 'cpu').
        n_epochs: Number of epochs to train for.
        train_loader: DataLoader for the training set.
        val_loader: DataLoader for the validation set.

    Returns:
        A tuple (model, history) where history is a dict containing
        per-epoch train_loss, train_acc, and val_acc lists.
    """
    loss_fcn = nn.CrossEntropyLoss()

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_acc": []
    }

    for epoch in range(n_epochs):
        # ----- Training phase -----
        model.train()
        running_loss = 0.0
        total = 0
        correct = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_fcn(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / total
        train_acc = correct / total

        # ----- Validation phase -----
        val_acc = evaluate_model(model, val_loader, device)

        # ----- Scheduler step -----
        # Handles both standard schedulers and ReduceLROnPlateau,
        # which needs a metric passed in.
        if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_acc)
        else:
            scheduler.step()

        # ----- Logging -----
        current_lr = optimizer.param_groups[0]["lr"]
        print(f"Epoch [{epoch + 1}/{n_epochs}] - "
              f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.4f} - "
              f"Val Acc: {val_acc:.4f} - LR: {current_lr:.6f}")

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

    return model, history

# Optuna Objective Function

In [31]:
def objective(trial, device, train_loader, val_loader):
    # Sample hyperparameters for the feature extractor
    n_layers = trial.suggest_int("n_layers", 1, 3)
    n_filters = [trial.suggest_int(f"n_filters_{i}", 16, 128) for i in range(n_layers)]
    kernel_sizes = [trial.suggest_categorical(f"kernel_size_{i}", [3, 5]) for i in range(n_layers)]

    # Sample hyperparameters for the classifier
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    fc_size = trial.suggest_int("fc_size", 64, 256)

    # Instantiate the model with the sampled hyperparameters
    model = CNN(n_layers, n_filters, kernel_sizes, dropout_rate, fc_size).to(device)

    # Initialize the dynamic classifier by passing a dummy input through the model
    dummy_input = torch.randn(1, 3, 32, 32).to(device)
    model(dummy_input)

    # Fixed training parameters
    learning_rate = 0.01
    loss_fcn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Train for a small number of epochs during search (keep search fast)
    n_epochs = trial.suggest_int("n_epochs", 3, 10)
    model = train_model(model, optimizer, train_loader, n_epochs, loss_fcn, device)

    # Evaluate on the validation set — this is what Optuna will optimize
    val_accuracy = evaluate_model(model, val_loader, device)

    return val_accuracy

In [ ]:
# Build the dataloaders once, outside the objective, so every trial reuses them
train_loader, val_loader, test_loader = get_data_loaders_with_validation()

print(len(train_loader.dataset))
print(len(val_loader.dataset))
print(len(test_loader.dataset))

45000
5000
300000


In [33]:
# Create an Optuna study to search for the best hyperparameters,
# maximizing the objective's return value (validation accuracy)
study = optuna.create_study(direction='maximize')

# Number of different hyperparameter combinations to try
n_trials = 20

# Run the optimization loop:
# - For each trial, Optuna calls objective(trial, device, train_loader, val_loader)
# - The lambda binds device/train_loader/val_loader so they're reused across trials
#   without being recreated on every call
# - Optuna uses the returned val_accuracy to decide which hyperparameters to try next
study.optimize(
    lambda trial: objective(trial, device, train_loader, val_loader),
    n_trials=n_trials
)

[I 2026-08-13 15:20:17,544] A new study created in memory with name: no-name-8659bb4d-87da-4c36-a50e-2c7ec30e214b


Epoch [1/3] - Loss: 0.0031 - Accuracy: 0.9989
Epoch [2/3] - Loss: 0.0000 - Accuracy: 1.0000
Epoch [3/3] - Loss: 0.0000 - Accuracy: 1.0000


[I 2026-08-13 15:23:42,847] Trial 0 finished with value: 1.0 and parameters: {'n_layers': 1, 'n_filters_0': 50, 'kernel_size_0': 5, 'dropout_rate': 0.4703180258839621, 'fc_size': 107, 'n_epochs': 3}. Best is trial 0 with value: 1.0.


Epoch [1/10] - Loss: 0.0031 - Accuracy: 0.9990
Epoch [2/10] - Loss: 0.0000 - Accuracy: 1.0000
Epoch [3/10] - Loss: 0.0000 - Accuracy: 1.0000
Epoch [4/10] - Loss: 0.0000 - Accuracy: 1.0000
Epoch [5/10] - Loss: 0.0000 - Accuracy: 1.0000


[W 2026-08-13 15:25:19,798] Trial 1 failed with parameters: {'n_layers': 2, 'n_filters_0': 53, 'n_filters_1': 87, 'kernel_size_0': 5, 'kernel_size_1': 3, 'dropout_rate': 0.3431202652195773, 'fc_size': 92, 'n_epochs': 10} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "d:\miniconda\envs\GradProject\lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\user\AppData\Local\Temp\ipykernel_12348\3069838130.py", line 5, in <lambda>
    lambda trial: objective(trial, device, train_loader, val_loader),
  File "C:\Users\user\AppData\Local\Temp\ipykernel_12348\2438388104.py", line 25, in objective
    model = train_model(model, optimizer, train_loader, n_epochs, loss_fcn, device)
  File "C:\Users\user\AppData\Local\Temp\ipykernel_12348\3230266982.py", line 9, in train_model
    for inputs, labels in train_dataloader:
  File "d:\miniconda\envs\GradProject\lib\site-packages\torch\utils

KeyboardInterrupt: 